# 🚀 Sentiment AI 2.0 (Pro Version) - High Precision & Big Data

Phiên bản nâng cấp sử dụng kỹ thuật **Back-translation (Dịch ngược)** để nhân bản dữ liệu và **PhoBERT-Large** để tối ưu độ chính xác.

**Cải tiến trong bản Pro 2.0:**
1. **Dữ liệu thông minh:** Dùng AI dịch VN -> EN -> VN để tạo ra hàng ngàn câu mới cùng ý nghĩa.
2. **Lõi AI mạnh mẽ:** Sử dụng `vinai/phobert-large` (370M parameters).
3. **Tối ưu Precision:** Tinh chỉnh để giảm thiểu việc đoán sai cảm xúc.

**Lưu ý:** 
- Bạn **PHẢI** chọn Runtime là **T4 GPU**.
- Thời gian huấn luyện dự kiến: ~1.5 - 2 giờ (bao gồm cả thời gian dịch dữ liệu).

In [ ]:
# 1. Cài đặt thư viện chuyên sâu
!pip install transformers[torch] datasets underthesea scikit-learn tqdm sacremoses

In [ ]:
# 2. Thu thập và Tăng cường Dữ liệu (Back-translation Augmentation)
from datasets import load_dataset
import pandas as pd
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

print("📥 Loading UIT-VSMEC dataset...")
vsmec = load_dataset("tridm/UIT-VSMEC")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("📡 Loading Translation Models (VN-EN and EN-VN)...")
# Sử dụng AutoModel thay vì pipeline để tránh lỗi KeyError
vi_en_model_name = "Helsinki-NLP/opus-mt-vi-en"
en_vi_model_name = "Helsinki-NLP/opus-mt-en-vi"

vi_en_tokenizer = AutoTokenizer.from_pretrained(vi_en_model_name)
vi_en_model = AutoModelForSeq2SeqLM.from_pretrained(vi_en_model_name).to(device)

en_vi_tokenizer = AutoTokenizer.from_pretrained(en_vi_model_name)
en_vi_model = AutoModelForSeq2SeqLM.from_pretrained(en_vi_model_name).to(device)

def translate(texts, model, tokenizer):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in outputs]

def back_translate(texts):
    try:
        # Dịch VN -> EN
        en_texts = translate(texts, vi_en_model, vi_en_tokenizer)
        # Dịch lại EN -> VN
        vn_back_texts = translate(en_texts, en_vi_model, en_vi_tokenizer)
        return vn_back_texts
    except Exception as e:
        print(f"⚠️ Lỗi dịch: {e}")
        return texts

def process_augmentation():
    os.makedirs("data", exist_ok=True)
    for split in ['train', 'validation', 'test']:
        df_orig = pd.DataFrame(vsmec[split])[['Sentence', 'Emotion']]
        
        if split == 'train':
            print(f"🔄 Augmenting {split} split (Back-translation)... Hãy kiên nhẫn...")
            batch_size = 16 # Giảm batch size cho an toàn bộ nhớ
            augmented_sentences = []
            sentences = df_orig['Sentence'].tolist()
            
            for i in tqdm(range(0, len(sentences), batch_size)):
                batch = sentences[i:i+batch_size]
                augmented_sentences.extend(back_translate(batch))
            
            df_aug = pd.DataFrame({
                'Sentence': augmented_sentences, 
                'Emotion': df_orig['Emotion'].tolist()
            })
            
            merged = pd.concat([df_orig, df_aug], ignore_index=True).drop_duplicates(subset=['Sentence'])
            print(f"✅ Augmented Training Data: {len(df_orig)} -> {len(merged)} samples")
        else:
            merged = df_orig
            
        merged.to_csv(f"data/{split}.csv", index=False, encoding='utf-8-sig')

process_augmentation()

In [ ]:
# 3. Huấn luyện PhoBERT-Large (High Precision Logic)
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score

device_train = "cuda" if torch.cuda.is_available() else "cpu"
EMOTIONS = ["Enjoyment", "Sadness", "Fear", "Anger", "Disgust", "Surprise", "Other"]
label2id = {label: i for i, label in enumerate(EMOTIONS)}
id2label = {i: label for i, label in enumerate(EMOTIONS)}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    prec = precision_score(labels, predictions, average='macro', zero_division=0)
    return {"accuracy": acc, "f1": f1, "precision": prec}

train_df = pd.read_csv("data/train.csv")
val_df = pd.read_csv("data/validation.csv")

class_weights = torch.tensor([1.0, 1.5, 5.0, 2.3, 3.0, 8.0, 1.0]).to(device_train)

class ProTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

model_name = "vinai/phobert-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=len(EMOTIONS), 
    id2label=id2label, 
    label2id=label2id
).to(device_train)

def tokenize_function(examples):
    return tokenizer(examples["Sentence"], padding="max_length", truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_tokenized = train_dataset.map(lambda x: {"label": label2id[x['Emotion']]}, batched=False).map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(lambda x: {"label": label2id[x['Emotion']]}, batched=False).map(tokenize_function, batched=True)

training_args = TrainingArguments(
    output_dir="pro_checkpoints",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=12,         
    weight_decay=0.02,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="precision", 
    fp16=True,
    report_to="none"
)

trainer = ProTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] 
)

print("🔥 Bắt đầu huấn luyện bản PRO (Augmented)... Vui lòng không tắt máy!")
trainer.train()

final_output = "final_model_pro"
model.save_pretrained(final_output)
tokenizer.save_pretrained(final_output)
print(f"✅ XONG! Model PRO đã sẵn sàng: {final_output}")

In [ ]:
# 4. Đóng gói Model
!zip -r final_model_pro.zip final_model_pro
from google.colab import files
files.download("final_model_pro.zip")